In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir('E:/Machine_Learning/Cases/Image_Segmentation')

In [13]:
img = pd.read_csv('Image_Segmentation.csv')
X,y = img.drop('Class',axis=1),img['Class']

In [14]:
y.unique()

array(['BRICKFACE', 'SKY', 'FOLIAGE', 'CEMENT', 'WINDOW', 'PATH', 'GRASS'],
      dtype=object)

In [26]:
clf = IsolationForest(contamination=0.05,random_state=26)
clf.fit(X)
predictions = clf.predict(X)
print("percentage of outliers = "+str((predictions<0).mean()*100)+"%")
ab_ind = np.where(predictions<0)[0]   # np.where returns tuple so we do this 
print('Outliers : ')
print(ab_ind)

percentage of outliers = 5.05050505050505%
Outliers : 
[ 32  35  42  62  63 105 108 121 139 182]


In [28]:
print(y[ab_ind])    ## if we don't do ab_ind = np.where(predictions<0)[0]  then we have to do print(y[X.index[ab_ind]])

32         SKY
35         SKY
42         SKY
62     FOLIAGE
63     FOLIAGE
105     CEMENT
108     CEMENT
121     WINDOW
139     WINDOW
182      GRASS
Name: Class, dtype: object


In [17]:
y[ab_ind].value_counts()

Class
SKY        5
FOLIAGE    4
CEMENT     1
PATH       1
Name: count, dtype: int64

In [32]:
# pd.crosstab(y,predictions)

## Removing outliers and applying algorithm on them

In [23]:
from sklearn.model_selection import StratifiedKFold,GridSearchCV
from sklearn.linear_model import LogisticRegression


In [18]:
img = img[predictions==1]   # predictions==-1 will be outliers
X,y = img.drop('Class',axis=1),img['Class']


In [21]:
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
lr = LogisticRegression()
lr.get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': 0.0,
 'max_iter': 100,
 'n_jobs': None,
 'penalty': 'deprecated',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

In [25]:
params = {
    'C' : np.linspace(0.001,5,10),
    'solver' : ['lbfgs','newton-cg','newton-cholesky','sag','saga']
}
gcv = GridSearchCV(lr,params,scoring='neg_log_loss',cv=kfold)
gcv.fit(X,y)
print('Best params : ',gcv.best_params_,"Best score : ",gcv.best_score_)

Best params :  {'C': np.float64(0.001), 'solver': 'newton-cg'} Best score :  -0.4760310518194501


## without outliers : 

In [30]:
img = pd.read_csv('Image_Segmentation.csv')
X,y = img.drop('Class',axis=1),img['Class']

params = {
    'C' : np.linspace(0.001,5,10),
    'solver' : ['lbfgs','newton-cg','newton-cholesky','sag','saga']
}
gcv = GridSearchCV(lr,params,scoring='neg_log_loss',cv=kfold)
gcv.fit(X,y)
print('Best params : ',gcv.best_params_,"Best score : ",gcv.best_score_)

Best params :  {'C': np.float64(0.5564444444444444), 'solver': 'newton-cholesky'} Best score :  -0.48076881996119897
